In [39]:
import sys
import os
sys.path.append(os.path.abspath('../../../'))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from functions.preproc import *
from functions.cox import *
from functions.repeated_cv import *

from sklearn import set_config
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import make_scorer
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler, StandardScaler
from sklearn.model_selection import StratifiedGroupKFold, StratifiedKFold
from sksurv.metrics import integrated_brier_score
from sksurv.metrics import concordance_index_ipcw
from sksurv.nonparametric import kaplan_meier_estimator
from sklearn.inspection import permutation_importance
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.ensemble import IsolationForest

from sksurv.metrics import concordance_index_censored
from sksurv.util import Surv
from sksurv.linear_model import CoxnetSurvivalAnalysis, CoxPHSurvivalAnalysis

import jinja2

import time
set_config(display="text")  # displays text representation of estimators

In [40]:
dataset = pd.read_csv("../../data/data_final.csv")

diagnosis_variables = []

with open("../../data/diagnosis_final.txt", "r") as file:
    for line in file:
        diagnosis_variables.append(line.strip())

treatment_variables = []
with open("../../data/treatment_final.txt", "r") as file:
    for line in file:
        treatment_variables.append(line.strip())

survival_dataset = dataset.drop('Unnamed: 0', axis=1)

feature_columns = treatment_variables + diagnosis_variables

In [41]:
# Find duplicated patients with regional ln recurrence = 1
mask = (survival_dataset.duplicated(subset='PERSON_ID', keep=False)) #& (survival_dataset['Regional LN recurrence'] == 1)

# For each duplicated PERSON_ID with recurrence, keep only the row with the lowest survival days
to_remove = (
    survival_dataset[mask]
    .sort_values(['PERSON_ID', 'Survival days'], ascending=[True, True])
    .groupby('PERSON_ID')
    .head(-1)
    .index
)

survival_dataset = survival_dataset.drop(to_remove)

Changing for the 5 years

In [42]:
survival_dataset = survival_dataset[
    ~((survival_dataset['Survival days'] > 1825) & (survival_dataset['Regional LN recurrence'] == 1))
]

In [44]:
# Prepare data
data_x = survival_dataset[feature_columns]

data_y = Surv.from_arrays(
    event=survival_dataset['Regional LN recurrence'],
    time=survival_dataset['Survival days']
)
# Since every duplicated patient has a consistent event indicator (regional ln recurrence) I can use this method
groups = survival_dataset['PERSON_ID']
event_indicator = survival_dataset['Regional LN recurrence']

In [45]:
sgkf = StratifiedKFold(n_splits=5, shuffle=True, random_state=12)
cv = sgkf.split(X=data_x, y=survival_dataset['Regional LN recurrence'])

train_index, test_index = next(cv)

In [46]:
x_train = data_x.iloc[train_index]
y_train = data_y[train_index]

x_test = data_x.iloc[test_index]
y_test = data_y[test_index]

subsampling = survival_dataset.iloc[train_index].copy()

In [47]:
# best comb for ipcw
alpha_range = [0.1]
l1_ratio_range = [0.5]
controls_per_case_range = [1]


metrics = ['c-index', 'c-index-ipcw', 'int-brier-score']

# Run grid search
results, summary_df, best_params = subsampling_repeated_grid_search(subsampling, x_train, y_train, alpha_range, l1_ratio_range, 
                                     controls_per_case_range, metrics, feature_columns,
                                     event_col='Regional LN recurrence', 
                                     time_col='Survival days',
                                     scale_data=True, scaler_type='standard',
                                     n_splits=4, n_repeats=5, base_random_state=42,
                                     verbose=True)

SUBSAMPLING REPEATED GRID SEARCH
Controls per case: [1]
L1 ratios: [0.5]
Alphas: 1 values from 1.00e-01 to 1.00e-01
Total hyperparameter combinations: 1
Folds per repetition: 4
Repetitions: 5
Total evaluations: 20
Scaling: Yes (standard)

[1/1] controls=1, l1=0.500, alpha=1.00e-01 (elapsed: 0.0min, est. remaining: 0.0min)
⚠️  Info: 6 constant features detected with StandardScaler.
   Removing these features to avoid NaN values (StandardScaler divides by std=0).
   Constant feature indices: [ 1 32 33 38 40 47]
   Features after removal: 42 (removed 6)
⚠️  Info: 6 constant features detected with StandardScaler.
   Removing these features to avoid NaN values (StandardScaler divides by std=0).
   Constant feature indices: [ 1  5 32 38 40 47]
   Features after removal: 42 (removed 6)
⚠️  Info: 7 constant features detected with StandardScaler.
   Removing these features to avoid NaN values (StandardScaler divides by std=0).
   Constant feature indices: [ 1 32 33 38 40 45 47]
   Features afte

In [50]:
summary_df.to_csv('cox_sub_5.csv', index=False)

In [ ]:
# Define hyperparameter ranges
alpha_range = np.logspace(-4, -1, 10)
l1_ratio_range = [1.0, 0.8, 0.5, 0.1, 0.01, 0.001]
controls_per_case_range = [1, 2, 4, 10, 15]


metrics = ['c-index', 'c-index-ipcw', 'int-brier-score']

# Run grid search
results, summary_df, best_params = subsampling_repeated_grid_search(subsampling, x_train, y_train, alpha_range, l1_ratio_range, 
                                     controls_per_case_range, metrics, feature_columns,
                                     event_col='Regional LN recurrence', 
                                     time_col='Survival days',
                                     scale_data=True, scaler_type='standard',
                                     n_splits=4, n_repeats=5, base_random_state=42,
                                     verbose=True)


SUBSAMPLING REPEATED GRID SEARCH
Controls per case: [1, 2, 4, 10, 15]
L1 ratios: [1.0, 0.8, 0.5, 0.1, 0.01, 0.001]
Alphas: 10 values from 1.00e-04 to 1.00e-01
Total hyperparameter combinations: 300
Folds per repetition: 4
Repetitions: 5
Total evaluations: 6000
Scaling: Yes (standard)

[1/300] controls=1, l1=1.000, alpha=1.00e-04 (elapsed: 0.0min, est. remaining: 0.1min)
⚠️  Info: 6 constant features detected with StandardScaler.
   Removing these features to avoid NaN values (StandardScaler divides by std=0).
   Constant feature indices: [ 1 32 33 38 40 47]
   Features after removal: 42 (removed 6)


c:\Users\Joao Goncalves\anaconda3\envs\survival-env\Lib\site-packages\sksurv\linear_model\coxph.py:125: RuntimeWarning: overflow encountered in exp
  risk_score = np.exp(linear_predictor)


⚠️  Info: 9 constant features detected with StandardScaler.
   Removing these features to avoid NaN values (StandardScaler divides by std=0).
   Constant feature indices: [ 1  5 32 33 34 40 41 45 47]
   Features after removal: 39 (removed 9)


c:\Users\Joao Goncalves\anaconda3\envs\survival-env\Lib\site-packages\sksurv\linear_model\coxph.py:125: RuntimeWarning: overflow encountered in exp
  risk_score = np.exp(linear_predictor)


⚠️  Info: 7 constant features detected with StandardScaler.
   Removing these features to avoid NaN values (StandardScaler divides by std=0).
   Constant feature indices: [ 1  5 32 33 38 44 47]
   Features after removal: 41 (removed 7)
⚠️  Info: 5 constant features detected with StandardScaler.
   Removing these features to avoid NaN values (StandardScaler divides by std=0).
   Constant feature indices: [ 1 32 33 38 47]
   Features after removal: 43 (removed 5)


c:\Users\Joao Goncalves\anaconda3\envs\survival-env\Lib\site-packages\sksurv\linear_model\coxph.py:129: RuntimeWarning: overflow encountered in power
  funcs[i] = StepFunction(x=self.baseline_survival_.x, y=np.power(self.baseline_survival_.y, risk_score[i]))


... (889 outputs left out)

c:\Users\Joao Goncalves\anaconda3\envs\survival-env\Lib\site-packages\sksurv\linear_model\coxph.py:125: RuntimeWarning: overflow encountered in exp
  risk_score = np.exp(linear_predictor)


⚠️  Info: 3 constant features detected with StandardScaler.
   Removing these features to avoid NaN values (StandardScaler divides by std=0).
   Constant feature indices: [32 33 38]
   Features after removal: 45 (removed 3)
⚠️  Info: 2 constant features detected with StandardScaler.
   Removing these features to avoid NaN values (StandardScaler divides by std=0).
   Constant feature indices: [32 33]
   Features after removal: 46 (removed 2)
[246/300] controls=15, l1=1.000, alpha=4.64e-03 (elapsed: 67.9min, est. remaining: 14.9min)
⚠️  Info: 2 constant features detected with StandardScaler.
   Removing these features to avoid NaN values (StandardScaler divides by std=0).
   Constant feature indices: [32 38]
   Features after removal: 46 (removed 2)
⚠️  Info: 2 constant features detected with StandardScaler.
   Removing these features to avoid NaN values (StandardScaler divides by std=0).
   Constant feature indices: [32 33]
   Features after removal: 46 (removed 2)
⚠️  Info: 1 constant 

c:\Users\Joao Goncalves\anaconda3\envs\survival-env\Lib\site-packages\sksurv\linear_model\coxph.py:125: RuntimeWarning: overflow encountered in exp
  risk_score = np.exp(linear_predictor)


⚠️  Info: 2 constant features detected with StandardScaler.
   Removing these features to avoid NaN values (StandardScaler divides by std=0).
   Constant feature indices: [32 33]
   Features after removal: 46 (removed 2)
⚠️  Info: 2 constant features detected with StandardScaler.
   Removing these features to avoid NaN values (StandardScaler divides by std=0).
   Constant feature indices: [32 33]
   Features after removal: 46 (removed 2)
⚠️  Info: 2 constant features detected with StandardScaler.
   Removing these features to avoid NaN values (StandardScaler divides by std=0).
   Constant feature indices: [33 47]
   Features after removal: 46 (removed 2)
⚠️  Info: 2 constant features detected with StandardScaler.
   Removing these features to avoid NaN values (StandardScaler divides by std=0).
   Constant feature indices: [33 47]
   Features after removal: 46 (removed 2)
⚠️  Info: 3 constant features detected with StandardScaler.
   Removing these features to avoid NaN values (Standard

In [19]:
summary_df.to_csv('../../results/repeated_sub/cox_sub.csv', index=False)

In [4]:
import pandas as pd

cox_sub = pd.read_csv('../../results/repeated_sub/cox_sub.csv')
cox_full = pd.read_csv('../../results/repeated_full/cox_full.csv')

In [ ]:
# ============================================================================
# CLEAN COX DATA: Remove rows with missing fold values (gaps)
# ============================================================================
def clean_cox_data(df):
    """
    Remove rows that have any NaN values in the c-index or c-index-ipcw fold columns.
    This ensures we only keep configurations where all folds computed successfully.
    """
    # Get all fold columns for c-index (normal)
    cindex_fold_cols = [col for col in df.columns if col.startswith('c-index_repeat_') and '_fold_' in col]
    
    # Get all fold columns for c-index-ipcw
    ipcw_fold_cols = [col for col in df.columns if 'c-index-ipcw_repeat_' in col and '_fold_' in col]
    
    # Combine all fold columns
    all_fold_cols = cindex_fold_cols + ipcw_fold_cols
    
    # Count rows before cleaning
    rows_before = len(df)
    
    # Keep only rows where ALL fold columns have valid (non-NaN) values
    df_clean = df.dropna(subset=all_fold_cols, how='any')
    
    rows_after = len(df_clean)
    rows_removed = rows_before - rows_after
    
    print(f"  Checked {len(cindex_fold_cols)} c-index fold columns and {len(ipcw_fold_cols)} c-index-ipcw fold columns")
    print(f"  Removed {rows_removed} rows with missing fold values ({rows_before} -> {rows_after})")
    
    return df_clean

print("\nCleaning Cox Full data...")
cox_full_clean = clean_cox_data(cox_full)

print("\nCleaning Cox Sub data...")
cox_sub_clean = clean_cox_data(cox_sub)

# Save cleaned data to CSV
print("\nSaving cleaned data to CSV...")
cox_full_clean.to_csv('../../results/repeated_full/cox_full_cleaned.csv', index=False)
cox_sub_clean.to_csv('../../results/repeated_sub/cox_sub_cleaned.csv', index=False)

print(f"\nSaved: ../../results/repeated_full/cox_full_cleaned.csv ({cox_full_clean.shape[0]} rows)")
print(f"Saved: ../../results/repeated_sub/cox_sub_cleaned.csv ({cox_sub_clean.shape[0]} rows)")
# Update the original variables with cleaned data
cox_full = cox_full_clean
cox_sub = cox_sub_clean


Cleaning Cox Full data...
  Checked 20 c-index fold columns and 20 c-index-ipcw fold columns
  Removed 20 rows with missing fold values (60 -> 40)

Cleaning Cox Sub data...
  Checked 20 c-index fold columns and 20 c-index-ipcw fold columns
  Removed 202 rows with missing fold values (300 -> 98)

Saving cleaned data to CSV...

Saved: ../../results/repeated_full/cox_full_cleaned.csv (40 rows)
Saved: ../../results/repeated_sub/cox_sub_cleaned.csv (98 rows)
